# savana.rainfall — end to-end test

Covers, in order: install/kernel sanity check, custom station inputs (single
coordinate, list, geojson/csv), preview-before-you-compute (stations map,
observations, product maps, GPCC overlay, inter-product bias, per-station
GPCC bias), date-range inheritance, extraction with caching, merge &
inspection (`compare_table`, `preview_comparison`), zone construction and
assignment (built from base regions, a single custom AOI, and the
lat-band fallback), formal validation (by zone / overall / thresholds),
ranking & application-weighted scoring, grounded insights
(`facts`/`summarize`/`answer`), matplotlib figures, the interactive Excel
decision workbook, and the one-call `validate_against_gpcc()` convenience
function. A full-scale (16 stations x 6 products x 2001-2020) run is at
the very end, clearly marked, since it's slow — everything above it uses
a small 1-2 station / 1-2 year scope so the notebook is fast to run
top to bottom.

**Before running**: set `EE_PROJECT` below to your actual Google Cloud
project id.


In [ ]:
EE_PROJECT = "ee-desmond"  # <-- set to your actual GCP project id
TEST_STATION = (-0.17, 5.56)  # Accra, used throughout for the small-scale examples


## 0. Sanity check — right kernel, right install

In [ ]:
import sys
print(sys.executable)  # should contain \envs\geospatial\

import savana
print("savana version:", savana.__version__)

import savana.rainfall as rf
print(rf)


In [ ]:
# Confirm you're on the current code, not a stale cached copy --
# should print (products_ic: dict, stations_df, cache_dir=None)
import inspect
import savana.rainfall.extraction as extraction
print(inspect.signature(extraction.extract_all_products))


## 1. Custom station inputs — every accepted format

`stations=` on `RainfallAssessment` accepts a DataFrame, a `.geojson`/`.csv`
path, a list of coordinates, or a single coordinate — same flexibility
everywhere, not just on the convenience function.


In [ ]:
from savana.rainfall.pipeline import RainfallAssessment
from savana.rainfall import config

# a) default -- the 16 WA GPCC stations from the paper
ra_default = RainfallAssessment(ee_project=EE_PROJECT)
print(ra_default.stations_df.shape, "stations (should be (16, 6))")
ra_default.stations_df.head()


In [ ]:
# b) a single coordinate -- exactly what we tested live earlier
ra_single = RainfallAssessment(stations=TEST_STATION, ee_project=EE_PROJECT)
print(ra_single.stations_df)


In [ ]:
# c) a list of coordinates, no ids given -- auto-generated station_id
ra_list = RainfallAssessment(
    stations=[(-1.5, 12.4), (2.1, 6.5), (-0.17, 5.56)],
    ee_project=EE_PROJECT,
)
print(ra_list.stations_df)


In [ ]:
# d) a list of (station_id, lon, lat) tuples -- your own ids, if you have them
ra_named = RainfallAssessment(
    stations=[("MY01", -1.5, 12.4), ("MY02", 2.1, 6.5)],
    ee_project=EE_PROJECT,
)
print(ra_named.stations_df)


In [ ]:
# e) a .geojson file of Point features -- station_id/station_name pulled
# from feature properties if present, auto-generated otherwise
import json as _json

geojson_path = "test_stations.geojson"
with open(geojson_path, "w") as f:
    _json.dump({
        "type": "FeatureCollection",
        "features": [
            {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-1.5, 12.4]},
             "properties": {"station_id": "GJ1", "station_name": "Bobo-Dioulasso area"}},
            {"type": "Feature", "geometry": {"type": "Point", "coordinates": [2.1, 6.5]},
             "properties": {}},
        ],
    }, f)

ra_geojson = RainfallAssessment(stations=geojson_path, ee_project=EE_PROJECT)
print(ra_geojson.stations_df)


In [ ]:
# f) a .csv file -- station_id, station_name, lon, lat, elevation_m, source
ra_default.stations_df.to_csv("test_stations.csv", index=False)
ra_csv = RainfallAssessment(stations="test_stations.csv", ee_project=EE_PROJECT)
print(ra_csv.stations_df.shape, "(should match the default 16)")


## 2. Preview — look before you compute

The one we'll actually carry through the rest of the notebook: a single
custom station, small and fast.


In [ ]:
rainfall = RainfallAssessment(stations=TEST_STATION, ee_project=EE_PROJECT)
rainfall.preview_stations()  # geemap -- is this actually where you think it is?


In [ ]:
rainfall.ingest(start="2020-01-01", end="2020-12-31")
rainfall.preview_map("CHIRPS", kind="daily")   # single product's spatial pattern


In [ ]:
rainfall.preview_map("CHIRPS", kind="annual")  # same, as annual total (mm/yr)


In [ ]:
# inter-product comparison -- NEVER a GPCC comparison, GPCC has no
# gridded form in this package. This mirrors the GEE app's Bias Map button.
rainfall.preview_map("CHIRPS", reference="GPM_IMERG")


In [ ]:
# .get_observations() now correctly inherits the 2020-2020 range from
# .ingest() above -- no need to repeat the dates
rainfall.get_observations(source="download")
rainfall.obs_df


In [ ]:
rainfall.preview_observations()  # matplotlib -- does the raw GPCC series look sane?


In [ ]:
# real GPCC point values overlaid on the CHIRPS raster, colored on the
# same scale -- click a dot (with the Inspector tool active) to compare
# the exact GPCC value against the raster's pixel value at that point
rainfall.preview_map("CHIRPS", show_gpcc=True)


## 3. Extraction (with caching) and merge

In [ ]:
rainfall.extract(cache_dir="savana_rainfall_data")
# re-running this cell reuses savana_rainfall_data/precip_extraction_<PRODUCT>.csv
# instead of re-hitting Earth Engine -- delete that folder to force a fresh pull
rainfall.sim_df.head()


In [ ]:
rainfall.merge()
rainfall.merged_df.head()


In [ ]:
# the plain "let me just look at the numbers" table -- GPCC alongside
# every product, one row per station-month
rainfall.compare_table()


In [ ]:
rainfall.preview_comparison()  # obs vs sim scatter, before any formal metric


In [ ]:
rainfall.preview_station_bias("CHIRPS")  # per-station bias vs REAL GPCC, on the map


## 4. Ecological zones — three ways to get zone geometry

Skippable: without any of this, `.validate()` just runs pooled
(unzoned). Included here to exercise all three paths.


In [ ]:
from savana.rainfall import zones

# a) a single custom AOI, no stratification needed -- e.g. one national park boundary
one_zone = zones.single_region_zone((-2.0, 5.0, 1.0, 8.0), zone_name="My Study Area")
print(one_zone)


In [ ]:
# b) build real zones from your own base regions + latitude-band splits --
# generic port of the GEE zone-delineation script. Skip this cell if you
# don't have base region assets of your own; it's here to show the shape
# of the call, not required for anything below.
custom_zone_defs = [
    {"zone_name": "North", "source_zone": "my_region", "lat_min": 5, "lat_max": 15},
    {"zone_name": "South", "source_zone": "my_region", "lat_min": -5, "lat_max": 5},
]
# zones_fc = zones.build_zones_from_bands(
#     base_zones={"my_region": "projects/your-project/assets/your_region"},
#     zone_defs=custom_zone_defs,
#     bounds=(-20, -5, 25, 25),
# )

# c) the documented latitude-band fallback -- zero configuration, always available
rainfall.assign_zones()  # use_default_if_none=False by default -> latitude fallback
rainfall.stations_df[["station_id", "lon", "lat", "zone"]]


## 5. Formal validation, thresholds, ranking, scoring

In [ ]:
rainfall.validate()
rainfall.validation_overall_df


In [ ]:
if rainfall.validation_by_zone_df is not None:
    display(rainfall.validation_by_zone_df)
else:
    print("Pooled only -- no zone column was assigned above for this station set.")


In [ ]:
rainfall.analyze_thresholds()
rainfall.threshold_df.head()


In [ ]:
rainfall.ranking_df if rainfall.ranking_df is not None else "run .validate() first"


In [ ]:
rainfall.score()
rainfall.scores_df


In [ ]:
from savana.rainfall import decision

prod, score = decision.best_product(rainfall.scores_df, "Drought early warning")
print(f"Best product for drought early warning: {prod} (score {score:.3f})")


## 6. Grounded insights — facts, summary, Q&A

In [ ]:
facts = rainfall.facts()
print(rainfall.summarize())


In [ ]:
print(rainfall.answer("which product is best for fire risk monitoring?"))


In [ ]:
print(rainfall.answer("what about drought early warning?"))


## 7. Figures

In [ ]:
rainfall.show("recommendation_heatmap")


In [ ]:
from savana.rainfall import viz

viz.application_ranking_bars(rainfall.scores_df, "Drought early warning")


In [ ]:
if rainfall.validation_by_zone_df is not None:
    viz.metric_heatmap(rainfall.validation_by_zone_df, metric="kge")
else:
    viz.zonal_boxplot(rainfall.validation_overall_df.assign(zone="pooled"), metric="kge")


## 8. Decision workbook export

In [ ]:
rainfall.export_workbook("test_decision_tool.xlsx")
print("Written: test_decision_tool.xlsx")


## 9. The one-call convenience function

Same result as sections 3-5 above, but the whole thing in a single call
with plain parameters -- no manual chaining.


In [ ]:
from savana.rainfall import validate_against_gpcc

result = validate_against_gpcc(
    stations=TEST_STATION,
    products=["CHIRPS", "GPM_IMERG"],
    start_year=2020,
    end_year=2020,
    ee_project=EE_PROJECT,
    cache_dir="savana_rainfall_data",
)
print(result.summarize())


## 10. Optional — AI agent over this assessment

Requires the `savana[agents]` extra and a model API key (e.g.
`ANTHROPIC_API_KEY` in your environment). Skip this section if you
haven't set that up.


In [ ]:
from savana.agents import SavanaGeoAgent

agent = SavanaGeoAgent(rainfall=rainfall, model="anthropic")
print(agent.ask("Summarize the rainfall assessment and recommend a product "
                 "for flood forecasting."))


## 11. Full-scale run — SLOW, optional

Reproduces the published study exactly: all 16 WA stations, all 6
products, 2001-2020. Only run this once everything above has passed --
this can take a while (full 20-year GPM-IMERG/MERRA-2 ingestion,
16-station extraction per product, full decision workbook).


In [ ]:
# Uncomment to run the full study-scale assessment.
# full = validate_against_gpcc(ee_project=EE_PROJECT, cache_dir="savana_rainfall_data_full")
# print(full.summarize())
# full.export_workbook("WA_Precipitation_Decision_Tool_reproduced.xlsx")
